In [17]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
# sns.set_style('whitegrid')
import sys
import scipy
import os
import glob
import random
import requests
import math
import time

from scipy.interpolate import interp1d
import numpy.polynomial.polynomial as poly
import csv

from scipy.optimize import curve_fit
from sklearn import linear_model
from sklearn.preprocessing import MinMaxScaler

from datetime import datetime
import matplotlib.colors as colors2
from matplotlib import colormaps as cm
import matplotlib.cm as cmx

import pickle

## custom functions for analysis
import my_functions

#### The EoH performance data can be downloaded from the UK data service
The link is here: https://datacatalogue.ukdataservice.ac.uk/studies/study/9050#documentation

In [2]:
performanceSummaryPath = '../9050_EoH_working_folder/csv/cleaned_1/eoh_summary_for_publication.csv'
perfSummary_df = pd.read_csv(performanceSummaryPath)

In [3]:
includedIDs = perfSummary_df['Property_ID'].iloc[np.where(perfSummary_df['Included_SPF_analysis']==True)[0]].values
print('There are %d included IDs'%len(includedIDs))

There are 545 included IDs


In [4]:
perfSummary_df = perfSummary_df[perfSummary_df['Property_ID'].isin(includedIDs)]

#### The information about the installations is available
The link is here: https://usmart.io/org/esc/discovery/discovery-view-detail/d506e975-cb39-47d7-99e0-e99939a2c4bf

In [5]:
houseMetaPath = '../9050_EoH_working_folder/mrdoc/excel/BEIS Electrification of Heat Project - Property, Design and Installation Information.csv'
meta_df = pd.read_csv(houseMetaPath)

In [6]:
meta_df = meta_df[meta_df['Property_ID'].isin(includedIDs)]
print('we have the meta data for %d houses'%len(meta_df))

we have the meta data for 545 houses


In [7]:
### include only air source heat pumps

# remove the hybrid units
Hybrid_IDs = meta_df.iloc[np.where(meta_df['HP_Installed']=='Hybrid')]['Property_ID'].values
print('Removing %d'%len(Hybrid_IDs) + ' hybrid heat pumps')
meta_df = meta_df[~meta_df['Property_ID'].isin(Hybrid_IDs)]
includedIDs = [x for x in includedIDs if x not in Hybrid_IDs]

# remove the ground source units too since there are so few
GSHP_IDs = meta_df.iloc[np.where(meta_df['HP_Installed']=='GSHP')]['Property_ID'].values
meta_df = meta_df[~meta_df['Property_ID'].isin(GSHP_IDs)]
includedIDs = [x for x in includedIDs if x not in GSHP_IDs]
print('Removing %d'%len(GSHP_IDs) + ' ground source heat pumps')

Removing 95 hybrid heat pumps
Removing 23 ground source heat pumps


In [8]:
# get a list of files for all houses
fileNameList = []
for j in range(4):
    filePath = '../9050_EoH_working_folder/csv/cleaned_'+str(j+1)+'/P*'
    nameList = glob.glob(filePath)
    for entry in nameList:
        fileNameList.append(entry)

In [9]:
print('There are files for %d'%len(fileNameList)+' households')

There are files for 742 households


In [10]:
# create a dict of the fileName and the key (EOHID)
fileNameDict = {}
for entry in fileNameList:
    key = entry.split('ID=')[1].split('.')[0]
    fileNameDict[key] = entry
fileNameDict = {k: v for k, v in fileNameDict.items() if k in includedIDs}

In [11]:
print('There are files for %d'%len(fileNameDict)+' households included in SPF analysis')

There are files for 427 households included in SPF analysis


In [12]:
# for the first house, print some summary stats
EoH_ID = includedIDs[1]
# the filename convention is as follows:
houseSummary_df = perfSummary_df[perfSummary_df['Property_ID']==EoH_ID]
row = houseSummary_df.iloc[0]
count = 0
for col in houseSummary_df.columns:
    print(f"{col}\t{row[col]}")
    count+=1
    if count>15:
        break

Property_ID	EOH0005
Included_SPF_analysis	True
Excluded_SPF_analysis_reason	nan
Whole_dataset_start	21/05/2021 14:00
Whole_dataset_end	28/09/2023 23:58
Whole_dataset_duration_days	860.42
Cleansed_dataset_start	21/05/2021 14:02
Cleansed_dataset_end	28/09/2023 23:56
Cleansed_dataset_duration_days	860.41
Selected_window_start	27/07/2022
Selected_window_end	27/07/2023
Max_quality_score_selected_window	2.0
Mean_quality_score_selected_window	0.001
SPFH2_selected_window	3.476
SPFH3_selected_window	3.476
SPFH4_selected_window	3.405


In [13]:
fileName = fileNameDict[EoH_ID]
print(pd.to_datetime( datetime.strptime(houseSummary_df['Selected_window_start'].iloc[0], "%d/%m/%Y") ))
print(fileName)

2022-07-27 00:00:00
../9050_EoH_working_folder/csv/cleaned_1/Property_ID=EOH0005.csv


In [14]:
# let's just make sure that we can calculate the SPF correctly
df = pd.read_csv(fileName, header=0)

# keep only the data for the selected window
start_TS = pd.to_datetime( datetime.strptime(houseSummary_df['Selected_window_start'].iloc[0], "%d/%m/%Y") )
end_TS = pd.to_datetime( datetime.strptime(houseSummary_df['Selected_window_end'].iloc[0], "%d/%m/%Y") )

# convert the df column to datetime
df['Timestamp'] = pd.to_datetime(df['Timestamp'])

mask = (df['Timestamp'] >= start_TS) & (df['Timestamp'] <= end_TS)
df = df.loc[mask,:]
df = df.set_index('Timestamp')

SPF_H2, SPF_H3, SPF_H4 = my_functions.get_SPFs(df)

print('SPF_H2 = %.2f' %SPF_H2)
print('SPF_H3 = %.2f' %SPF_H3)
print('SPF_H4 = %.2f' %SPF_H4)
print( 'The heat pump size is %.2f'%meta_df[meta_df['Property_ID']==EoH_ID]['HP_Size_kW'].values[0] )

SPF_H2 = 3.48
SPF_H3 = 3.48
SPF_H4 = 3.41
The heat pump size is 7.00


In [18]:
# # import the prices from the different regions from Octopus for the agile tariff

# PRODUCT = "AGILE-18-02-21"
# TARIFF_PREFIX = "E-1R-AGILE-18-02-21-{}"
# BASE = (
#     f"https://api.octopus.energy/v1/products/{PRODUCT}/"
#     f"electricity-tariffs/{TARIFF_PREFIX}/standard-unit-rates/"
# )

# regions = [chr(c) for c in range(ord("A"), ord("P") + 1) if chr(c) not in {"I", "O"}]

# params0 = {
#     "period_from": "2020-12-01T00:00Z",
#     "period_to": "2023-09-30T23:30Z",
#     "page_size": 250
# }

# def fetch_region_series(region, session=None):
#     s = session or requests.Session()
#     base_url = BASE.format(region)

#     all_results = []
#     url = base_url
#     params = params0.copy()

#     print(f"Fetching region {region}")

#     while url:
#         r = s.get(url, params=params)
#         if r.status_code == 429:
#             retry_after = r.headers.get("Retry-After")
#             wait = int(retry_after) if retry_after and retry_after.isdigit() else 5
#             time.sleep(wait)
#             continue

        
#         r.raise_for_status()
#         data = r.json()

#         all_results.extend(data["results"])

#         url = data["next"]
#         time.sleep(0.2)
#         params = None  # next URL already contains params

#     idx = pd.to_datetime([x["valid_from"] for x in all_results], utc=True)
#     vals = [x["value_inc_vat"] for x in all_results]

#     return pd.Series(vals, index=idx, name=region).sort_index()

# # --- Build DataFrame ---
# with requests.Session() as sess:
#     prices_df = pd.concat(
#         [fetch_region_series(r, session=sess) for r in regions],
#         axis=1
#     ).sort_index()

# print(prices_df.head())


Fetching region A
Fetching region B
Fetching region C
Fetching region D
Fetching region E
Fetching region F
Fetching region G
Fetching region H
Fetching region J
Fetching region K
Fetching region L
Fetching region M
Fetching region N
Fetching region P
                                A      B      C       D       E       F  \
2020-12-01 00:00:00+00:00  8.1585  7.770  7.770  8.5470  8.1585  8.1585   
2020-12-01 00:30:00+00:00  8.4840  8.085  8.085  8.8935  8.4840  8.4840   
2020-12-01 01:00:00+00:00  8.6415  8.232  8.232  9.0510  8.6415  8.6415   
2020-12-01 01:30:00+00:00  8.2950  7.896  7.896  8.6835  8.2950  8.2950   
2020-12-01 02:00:00+00:00  9.1770  8.736  8.736  9.6075  9.1770  9.1770   

                                G       H       J       K        L      M  \
2020-12-01 00:00:00+00:00  8.1585  8.1585  8.5470  8.5470   8.9355  7.770   
2020-12-01 00:30:00+00:00  8.4840  8.4840  8.8935  8.8935   9.3030  8.085   
2020-12-01 01:00:00+00:00  8.6415  8.6415  9.0510  9.0510   9.4710

In [25]:
# avg_prices_df = prices_df.mean(axis=1).to_frame(name="avg_price")
# avg_prices_df.to_parquet("average_agile_tariff_UK.parquet")
# del prices_df
avg_prices_df = pd.read_parquet("average_agile_tariff_UK.parquet")

In [24]:
avg_prices_df.head()

,avg_price
2020-12-01 00:00:00+00:00,8.29725
2020-12-01 00:30:00+00:00,8.63175
2020-12-01 01:00:00+00:00,8.78925
2020-12-01 01:30:00+00:00,8.43300
2020-12-01 02:00:00+00:00,9.33000


In [31]:
keySelect = random.sample(list(fileNameDict.keys()), 2)
for key in list(fileNameDict.keys()):
    if key not in keySelect:
        del fileNameDict[key]
# now debug with only two entries

In [33]:
fileNameDict

{'EOH0668': '../9050_EoH_working_folder/csv/cleaned_1/Property_ID=EOH0668.csv',
 'EOH1925': '../9050_EoH_working_folder/csv/cleaned_4/Property_ID=EOH1925.csv'}

In [24]:
ID_heat_elec = {}
counter = 0
for idx, fileName in fileNameDict.items():

    houseSummary = perfSummary_df[perfSummary_df['Property_ID']==EoH_ID]
    df = pd.read_csv(fileName, header=0)

    ### keep only the data for the selected window
    start_TS = pd.to_datetime( datetime.strptime(houseSummary['Selected_window_start'].iloc[0], "%d/%m/%Y") )
    end_TS = pd.to_datetime( datetime.strptime(houseSummary['Selected_window_end'].iloc[0], "%d/%m/%Y") )
    # convert the df column to datetime
    df['Timestamp'] = pd.to_datetime(df['Timestamp'])
    mask = (df['Timestamp'] >= start_TS) & (df['Timestamp'] <= end_TS)
    df = df.loc[mask,:]
    df = df.set_index('Timestamp')

    ### get SPFs
    SPF_H2, SPF_H3, SPF_H4 = my_functions.get_SPFs(df)
    ID_heat_elec[EoH_ID]['SPF_2'] = SPF_H2
    ID_heat_elec[EoH_ID]['SPF_3'] = SPF_H3
    ID_heat_elec[EoH_ID]['SPF_4'] = SPF_H4

    ### get the stuff we want out of the meta data
    HP_cap = meta_df[meta_df['Property_ID']==EoH_ID]['HP_Size_kW'].values[0]
    HP_cap_2min = (2*HP_cap)/30
    ID_heat_elec[EoH_ID]['HP_cap'] = HP_cap
    ID_heat_elec[EoH_ID]['postcode'] = meta_df[meta_df['Property_ID']==EoH_ID]['Postcode_1']
        
    # write conditionals for the immersion heater and backup heater  
    HP = df['Heat_Pump_Energy_Output'].diff()
    WS = df['Whole_System_Energy_Consumed'].diff()
    CP = df['Circulation_Pump_Energy_Consumed'].diff()
    
    IH = df['Immersion_Heater_Energy_Consumed'].diff() if 'Immersion_Heater_Energy_Consumed' in df.columns else 0
    BU = df['Back-up_Heater_Energy_Consumed'].diff() if 'Back-up_Heater_Energy_Consumed' in df.columns else 0
    HP_electric = WS - IH - BU 

    # create a new dataframe that stores internal air, external air, flow, return, heat out and elec in
    cols_for_temps = [
        'Internal_Air_Temperature',
        'External_Air_Temperature',
        'Flow_Temperature',
        'Return_Temperature',
    ]

    # adjust the names above to match your actual column names
    new_df = df[cols_for_temps].copy()
    new_df['Heat_out']   = HP          # heat output per interval (kWh or whatever unit)
    new_df['Elec_in']    = HP_electric # electrical input per interval
    # remove first NaN row due to the diffs:
    new_df = new_df.iloc[1:, :]

    # store this per-house dataframe in your dict
    # ID_heat_elec[EoH_ID]['timeseries'] = new_df
    

,Circulation_Pump_Energy_Consumed,External_Air_Temperature,Heat_Pump_Energy_Output,Heat_Pump_Heating_Flow_Temperature,Heat_Pump_Return_Temperature,Hot_Water_Flow_Temperature,Immersion_Heater_Energy_Consumed,Internal_Air_Temperature,Whole_System_Energy_Consumed
Timestamp,,,,,,,,,
2022-07-27 00:00:00,95.096,17.50,5824.013,25.09,23.40,NaN,2.697,21.64,2503.925
2022-07-27 00:02:00,95.096,17.47,5824.013,25.05,23.38,NaN,2.697,21.64,2503.926
2022-07-27 00:04:00,95.096,17.43,5824.013,25.00,23.35,NaN,2.697,21.64,2503.927
2022-07-27 00:06:00,95.096,17.40,5824.013,24.95,23.34,NaN,2.697,21.64,2503.927
2022-07-27 00:08:00,95.096,17.37,5824.013,24.90,23.31,NaN,2.697,21.64,2503.928
...,...,...,...,...,...,...,...,...,...
2023-07-26 23:52:00,170.552,16.69,10407.368,23.61,22.79,NaN,22.586,NaN,4555.470
2023-07-26 23:54:00,170.552,16.68,10407.368,23.60,22.76,NaN,22.586,NaN,4555.471
2023-07-26 23:56:00,170.552,16.66,10407.368,23.57,22.74,NaN,22.586,NaN,4555.472
